# ML-10 — Content Action Playbook

This notebook turns the validated model output from Weeks 4–6 into a **content action playbook** —
a ranked, human-reviewable queue of content actions with reason codes, known limits, and
monitoring triggers. The playbook is the recommendations section of the deployed research paper.

> **Skills loaded:** `writing-honest-claims` + `flyrank/flyrank-data`
>
> **Source model:** Week-5 XGBoost (GroupKFold by client, OOF AUC = 0.694)
>
> **Source baseline:** Week-4 rule score (visible × (stale × log(impr) + slipping × log(impr) × 0.5))
>
> **Claim ladder:** All language uses observed / directional / decision-support.
> Cross-sectional data never supports "doing X *will* produce Y."

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The playbook combines two scoring systems:
1. **Rule baseline score** (w04): transparent, hand-crafted, interpretable — `visible × (stale × log(impr) + slipping × log(impr) × 0.5)`
2. **XGBoost model probability** (w05): out-of-fold GroupKFold predictions (OOF AUC ≈ 0.694)

The final queue ranks pages by the model's predicted decline probability (higher = review first),
with the baseline reason code attached for human-readable context. The model was **observed** to
provide better ranking at depth than the rule baseline (w05), while the reason codes explain
*why* in words a content editor can act on.

### Archetype → action mapping

Based on observed patterns in the 30k-row dataset:

| Archetype | Observed pattern | Suggested action | Reason code |
|---|---|---|---|
| **Stale + slipping** | Updated ≥180d ago, position >10, visible | Full refresh + reposition | `stale_and_slipping` |
| **Stale + visible** | Updated ≥180d ago, position ≤10, visible | Content refresh | `stale_visible` |
| **Position slipping** | Recently updated, position >10, visible | On-page SEO optimization | `position_slipping` |
| **Visible only** | Visible, recently updated, position OK | Monitor — no action needed | `visible_only` |
| **Low visibility** | <500 impressions/90d | Deprioritize — low ROI for refresh | `low_visibility` |

### Decay/refresh insight

We **observed** that decline rate rises from 51.1% (0–30 day freshness) to 61.1% (91–180 days),
then drops to 47.1% at 181+ days (n=174). The non-monotonic pattern suggests that extremely stale
pages may have already stabilized at a low traffic floor. The actionable window for refresh appears
to be the 30–180 day range, where staleness is **associated with** higher decline rates but pages
still have meaningful traffic to preserve.

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

# ── Load data (same pipeline as w05/w06) ──
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# ── Feature engineering (identical to w05) ──
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_position"] = (df["avg_position"] > 0).astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["avg_position_clean"] = df["avg_position"].replace(0, np.nan)

log_cols = ["impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
            "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d"]
for col in log_cols:
    df[f"log_{col}"] = np.log1p(df[col])

NUMERIC_FEATURES = [
    "content_age_days", "days_since_last_update",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position_clean", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "search_volume", "competition", "cpc",
    "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_pageviews_90d",
    "log_sessions_90d", "log_users_90d", "log_engaged_sessions_90d",
    "log_ai_sessions_90d", "log_scroll_events_90d",
    "has_word_count", "has_position", "has_keyword_data",
]
CAT_FEATURES = ["content_type", "main_intent"]

df_model = pd.get_dummies(df, columns=CAT_FEATURES, drop_first=False, dtype=int)
onehot_cols = [c for c in df_model.columns
               if any(c.startswith(f"{cat}_") for cat in CAT_FEATURES)]
FEATURE_COLS = NUMERIC_FEATURES + onehot_cols

X = df_model[FEATURE_COLS].fillna(0).values
y = df_model["is_declining"].values
groups = df_model["client_id"].values
base_rate = y.mean()

print(f"Dataset: {len(df):,} rows x {len(FEATURE_COLS)} features")
print(f"Base rate: {base_rate*100:.1f}% declining")
print(f"Clients: {df['client_id'].nunique()}")

Dataset: 30,000 rows x 32 features
Base rate: 54.2% declining
Clients: 32


In [2]:
# ── Reproduce model OOF predictions (same as w05/w06) ──
oof_xgb = np.zeros(len(X))
gkf = GroupKFold(n_splits=5)

for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    X_tr, X_te = X[train_idx], X[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]

    neg = (y_tr == 0).sum()
    pos = (y_tr == 1).sum()
    model = XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.1,
        scale_pos_weight=neg / pos,
        random_state=SEED, n_jobs=-1, eval_metric="logloss",
        verbosity=0
    )
    model.fit(X_tr, y_tr)
    oof_xgb[test_idx] = model.predict_proba(X_te)[:, 1]

oof_auc = roc_auc_score(y, oof_xgb)
print(f"OOF AUC (GroupKFold by client): {oof_auc:.3f}")

# ── Reproduce baseline rule score (same as w04) ──
df_model["visible"] = (df_model["impressions_90d"] >= 500).astype(int)
df_model["stale"] = (df_model["days_since_last_update"] >= 180).astype(int)
df_model["slipping"] = (
    (df_model["avg_position"] > 10) & (df_model["avg_position"] > 0)
).astype(int)
baseline_scores = df_model["visible"].values * (
    df_model["stale"].values * np.log1p(df_model["impressions_90d"].values)
    + df_model["slipping"].values * np.log1p(df_model["impressions_90d"].values) * 0.5
)

OOF AUC (GroupKFold by client): 0.694


In [3]:
# ── Assign reason codes (same logic as w04) ──
def assign_reason(row):
    if row["visible"] == 0:
        return "low_visibility"
    if row["stale"] and row["slipping"]:
        return "stale_and_slipping"
    if row["stale"]:
        return "stale_visible"
    if row["slipping"]:
        return "position_slipping"
    return "visible_only"

df_model["reason_code"] = df_model.apply(assign_reason, axis=1)

# Map reason codes to suggested actions
action_map = {
    "stale_and_slipping": "refresh_and_reposition",
    "stale_visible": "refresh",
    "position_slipping": "optimize_position",
    "visible_only": "monitor",
    "low_visibility": "deprioritize",
}
df_model["suggested_action"] = df_model["reason_code"].map(action_map)

# Assign model probability and rank by it
df_model["model_prob"] = oof_xgb
df_model["model_rank"] = df_model["model_prob"].rank(method="first", ascending=False).astype(int)
df_model["baseline_score"] = baseline_scores

print("RANKED ACTION QUEUE -- top 20 by model probability")
print("=" * 100)

display_cols = ["model_rank", "content_id", "model_prob", "reason_code",
                "suggested_action", "impressions_90d", "avg_position",
                "days_since_last_update", "content_age_days", "is_declining"]
top20 = df_model.sort_values("model_rank").head(20)
print(top20[display_cols].to_string(index=False))

print()
print("Reason code distribution (full dataset):")
print(df_model["reason_code"].value_counts().to_string())

print()
print("Action distribution (full dataset):")
print(df_model["suggested_action"].value_counts().to_string())

RANKED ACTION QUEUE -- top 20 by model probability
 model_rank           content_id  model_prob       reason_code  suggested_action  impressions_90d  avg_position  days_since_last_update  content_age_days  is_declining
          1 content_20c5d6a1c7b1    0.984567    low_visibility      deprioritize              201           6.2                       8                91             1
          2 content_9d35d489e74d    0.983852    low_visibility      deprioritize              169           6.7                       8                96             1
          3 content_4f5826036689    0.981489 position_slipping optimize_position             9761          21.9                     104               153             1
          4 content_f57df93d1a2a    0.981192 position_slipping optimize_position             2643          35.0                     104               148             1
          5 content_19167461a1a4    0.978930 position_slipping optimize_position            16667          38

In [4]:
# ── Precision@K: how good is the ranked queue? ──
def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    return y_true[order[:k]].mean()

ks = [10, 20, 50, 100, 200]

print("RANKED QUEUE QUALITY -- Precision@K")
print("=" * 70)
print(f"{'Method':<25s}", end="")
for k in ks:
    print(f"  P@{k:<4d}", end="")
print("    AUC")
print("-" * 70)

# Base rate
row = f"{'Base rate':<25s}"
for k in ks:
    row += f"  {base_rate*100:5.1f}%"
row += "      --"
print(row)

# Rule baseline
row = f"{'Rule baseline (w04)':<25s}"
for k in ks:
    p = precision_at_k(y, baseline_scores, k)
    row += f"  {p*100:5.1f}%"
row += "      --"
print(row)

# XGBoost (the queue we export)
row = f"{'XGBoost queue (w05)':<25s}"
for k in ks:
    p = precision_at_k(y, oof_xgb, k)
    row += f"  {p*100:5.1f}%"
row += f"  {oof_auc:.3f}"
print(row)

print()
print("The XGBoost queue is observed to outperform the rule baseline at depths")
print("beyond 20, while the rule baseline concentrates its precision at the very")
print("top (P@10=100%). Both are decision-support tools -- not ground truth.")

RANKED QUEUE QUALITY -- Precision@K
Method                     P@10    P@20    P@50    P@100   P@200     AUC
----------------------------------------------------------------------
Base rate                   54.2%   54.2%   54.2%   54.2%   54.2%      --
Rule baseline (w04)        100.0%   80.0%   64.0%   58.0%   56.5%      --
XGBoost queue (w05)         80.0%   80.0%   78.0%   75.0%   74.0%  0.694

The XGBoost queue is observed to outperform the rule baseline at depths
beyond 20, while the rule baseline concentrates its precision at the very
top (P@10=100%). Both are decision-support tools -- not ground truth.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This playbook is a **decision-support tool for content teams** at FlyRank (or similar agencies)
who need to prioritize which pages to review for refresh, repositioning, or deprecation.

**What it does:**
- Produces a ranked list of content items sorted by predicted likelihood of decline
- Attaches a human-readable reason code explaining *why* each page was flagged
- Maps each reason code to a suggested content action

**What it does NOT do:**
- Predict Google's algorithm or guarantee that acting on a page will reverse decline
- Replace human editorial judgment about content quality, brand strategy, or seasonality
- Provide causal evidence that any feature *causes* decline

### Known limits

| Limit | Detail |
|---|---|
| **Cross-sectional snapshot** | The model was trained on one 90-day window. Temporal patterns (seasonality, algorithm updates) are not captured. |
| **32 clients only** | The training data covers 32 pseudonymized clients. Per-fold AUC varies substantially, indicating uneven model skill across client profiles. |
| **Partial window overlap** | `days_with_impressions` (top feature, importance 0.2255) shares a measurement window with the label. The w06 audit measured a modest AUC drop without it (0.694 → 0.669) — a disclosed dependency. |
| **Label definition** | "Declining" = impressions dropped ≥20% (last 30d vs. prev 30d). This threshold is arbitrary; a 19% drop would be classified as stable. |
| **No content quality signal** | The features are quantitative (traffic, position, age). Actual content quality, SERP competitor strength, and editorial intent are invisible to the model. |
| **Base rate is high** | At 54.2% declining, a coin flip is already competitive. The model's value is in *ranking*, not classification. |

In [5]:
# ── Quantify the limits with numbers ──
print("KNOWN LIMITS -- quantified")
print("=" * 70)

# 1. Per-fold AUC variance
fold_aucs = []
gkf2 = GroupKFold(n_splits=5)
for fold_idx, (train_idx, test_idx) in enumerate(gkf2.split(X, y, groups)):
    fold_auc = roc_auc_score(y[test_idx], oof_xgb[test_idx])
    fold_aucs.append(fold_auc)
    n_clients = len(set(groups[test_idx]))
    print(f"  Fold {fold_idx}: AUC={fold_auc:.3f} ({n_clients} test clients)")

print(f"\n  Overall OOF AUC: {oof_auc:.3f}")
print(f"  Fold range: {min(fold_aucs):.3f}-{max(fold_aucs):.3f}")
print(f"  Fold std: {np.std(fold_aucs):.3f}")
print()

# 2. Label threshold sensitivity
# trend_pct is already in percentage points (e.g. -41.4 means -41.4%)
# trend_direction == 'down' corresponds to trend_pct <= -20
# Some rows have NaN trend_pct (n=3,388) -- these are 'new' or 'flat' content
has_trend = df["trend_pct"].notna()
print(f"  Label threshold sensitivity (n={has_trend.sum():,} rows with trend_pct):")
for threshold_pct in [10, 15, 20, 25, 30, 50]:
    alt_declining = (df.loc[has_trend, "trend_pct"] <= -threshold_pct).sum()
    alt_rate = alt_declining / len(df)  # fraction of ALL rows
    print(f"    If >={threshold_pct}% drop = 'declining': {alt_declining:,} rows = {alt_rate*100:.1f}% of all pages")

print()
print("  The 20% threshold is a design choice, not a natural boundary.")
print("  Different thresholds shift both the base rate and model performance.")
print(f"  Additionally, {(~has_trend).sum():,} rows have no trend_pct (new/flat content)")
print(f"  -- the label groups them as non-declining via trend_direction != 'down'.")

KNOWN LIMITS -- quantified
  Fold 0: AUC=0.648 (1 test clients)
  Fold 1: AUC=0.605 (7 test clients)
  Fold 2: AUC=0.706 (8 test clients)
  Fold 3: AUC=0.698 (8 test clients)
  Fold 4: AUC=0.697 (8 test clients)

  Overall OOF AUC: 0.694
  Fold range: 0.605-0.706
  Fold std: 0.039

  Label threshold sensitivity (n=26,612 rows with trend_pct):
    If >=10% drop = 'declining': 18,274 rows = 60.9% of all pages
    If >=15% drop = 'declining': 17,358 rows = 57.9% of all pages
    If >=20% drop = 'declining': 16,313 rows = 54.4% of all pages
    If >=25% drop = 'declining': 15,263 rows = 50.9% of all pages
    If >=30% drop = 'declining': 14,138 rows = 47.1% of all pages
    If >=50% drop = 'declining': 9,646 rows = 32.2% of all pages

  The 20% threshold is a design choice, not a natural boundary.
  Different thresholds shift both the base rate and model performance.
  Additionally, 3,388 rows have no trend_pct (new/flat content)
  -- the label groups them as non-declining via trend_direct

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review rules

Before acting on any recommendation from this queue, a content editor **must** verify:

1. **Seasonality check:** Is this page's traffic naturally cyclical? A travel page declining in
   winter is not a content quality problem. The model has no seasonality signal.

2. **Recent external events:** Has there been a Google algorithm update, domain migration, or
   redirect change since the snapshot? The model's 90-day window is frozen.

3. **Content intent alignment:** Does the page still match the keyword intent it ranks for?
   The model sees position and traffic, not semantic relevance.

4. **Client strategy context:** Is this page being deliberately sunset, consolidated, or
   redirected? The model doesn't know business intent.

5. **Small-bucket caution:** If the reason code comes from `stale_and_slipping` (n=14 in the
   dataset) or `stale_visible` (n=3), the archetype has very few training examples.
   Treat these recommendations with extra skepticism.

### The no-go list — what should NOT be automated

| Never automate | Why |
|---|---|
| **Automatic content deletion** | A high model probability does not mean the page has zero value. It means the page *looks like* declining pages in this dataset. |
| **Automatic refresh/rewrite** | Refreshing content without human review risks destroying well-performing pages that the model misclassified (thousands of false positives in OOF evaluation). |
| **Client-facing reports without review** | The model's per-client accuracy varies widely. Presenting unreviewed scores to a specific client may be misleading. |
| **Cross-client generalization** | This model was trained on 32 clients. Deploying it on a new client's portfolio without validation on that client's data is not supported. |
| **Causal claims in any communication** | The model observes associations. Saying "this page is declining *because* it's stale" exceeds the evidence. |

In [6]:
# ── Quantify the false positive/negative risk that human review mitigates ──
print("ERROR RATES THAT MOTIVATE HUMAN REVIEW")
print("=" * 70)

pred_declining = (oof_xgb >= 0.5).astype(int)
tp = ((pred_declining == 1) & (y == 1)).sum()
fp = ((pred_declining == 1) & (y == 0)).sum()
tn = ((pred_declining == 0) & (y == 0)).sum()
fn = ((pred_declining == 0) & (y == 1)).sum()
acc = (tp + tn) / len(y)

print(f"  Accuracy: {acc*100:.1f}% (base rate: {base_rate*100:.1f}%)")
print(f"  True positives:  {tp:>6,}  |  False positives: {fp:>6,}")
print(f"  True negatives:  {tn:>6,}  |  False negatives: {fn:>6,}")
print()
print(f"  False positive rate: {fp/(fp+tn)*100:.1f}%")
print(f"  False negative rate: {fn/(fn+tp)*100:.1f}%")
print()
print(f"  -> {fp:,} pages would be flagged for review that are NOT declining.")
print("    This is why automated action on flagged pages is a no-go.")
print()

# ── Small-bucket warning ──
print("SMALL-BUCKET WARNING")
print("-" * 70)
for code in ["stale_and_slipping", "stale_visible"]:
    n = (df_model["reason_code"] == code).sum()
    decline_rate = df_model.loc[df_model["reason_code"] == code, "is_declining"].mean()
    print(f"  '{code}': n={n}, decline rate={decline_rate*100:.1f}%")
print("  These archetypes have very few examples -- treat with extra caution.")

ERROR RATES THAT MOTIVATE HUMAN REVIEW
  Accuracy: 64.8% (base rate: 54.2%)
  True positives:  11,656  |  False positives:  5,948
  True negatives:   7,790  |  False negatives:  4,606

  False positive rate: 43.3%
  False negative rate: 28.3%

  -> 5,948 pages would be flagged for review that are NOT declining.
    This is why automated action on flagged pages is a no-go.

SMALL-BUCKET WARNING
----------------------------------------------------------------------
  'stale_and_slipping': n=14, decline rate=92.9%
  'stale_visible': n=3, decline rate=100.0%
  These archetypes have very few examples -- treat with extra caution.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### When to retrain or re-evaluate

This model is a **snapshot**: it was trained on one 90-day window from 32 clients. It will go
stale. Here are the concrete triggers for re-evaluation:

| Trigger | How to detect | Action |
|---|---|---|
| **P@50 drops below 70%** | Re-run the queue on a new 90-day snapshot and measure P@K against ground truth | Retrain on updated data |
| **Base rate shifts >5 percentage points** | Compare new snapshot's decline rate to 54.2% | Retrain — the label distribution has changed |
| **New client onboarded** | Any client not in the training set of 32 | Validate on new client's data before trusting predictions |
| **Google algorithm update** | Industry news + sudden metric shifts across portfolio | Pause recommendations, collect post-update data, retrain |
| **Feature distribution drift** | Compare new snapshot's feature quantiles to training data | Investigate — model may extrapolate poorly |
| **>90 days since last training** | Calendar check | Re-evaluate whether the model's temporal window is still representative |

### Cost/value thinking

The playbook's value is **time saved in triage**, not in prediction accuracy.

- **Cost of a false positive (flagging a healthy page):** A content editor spends ~15 minutes
  reviewing a page that doesn't need work. The code output above quantifies the FP count.
- **Cost of a false negative (missing a declining page):** The page continues to decline
  unnoticed. The code output above quantifies the FN count.
- **Value of the queue:** The P@K table above shows how many true decliners the queue surfaces
  compared to random ordering at the 54.2% base rate. A content team can review the top 50 in
  a morning and catch substantially more declining pages than random triage.
- **Break-even:** If reviewing one page costs ~15 min of editor time, and fixing a declining page
  is worth one day of recovered traffic, the queue pays for itself if even a few of the
  top-50 catches lead to successful refreshes.

In [7]:
# ── Quantify cost/value with the observed P@K numbers ──
print("COST/VALUE ANALYSIS")
print("=" * 70)

for k in [10, 20, 50, 100, 200]:
    p_model = precision_at_k(y, oof_xgb, k)
    p_random = base_rate
    extra_catches = k * (p_model - p_random)
    print(f"  Top-{k:>3d}: P@K={p_model*100:.0f}% -> ~{k*p_model:.0f} true decliners "
          f"(vs ~{k*p_random:.0f} random) -> +{extra_catches:.0f} extra catches")

print()
print(f"  At 15 min/review, top-50 costs ~12.5 editor-hours.")
print(f"  The queue is observed to surface more true decliners than random")
print(f"  ordering at the same depth. Whether this justifies the review cost")
print(f"  depends on the value of catching decline early -- a business decision,")
print(f"  not a model decision.")

COST/VALUE ANALYSIS
  Top- 10: P@K=80% -> ~8 true decliners (vs ~5 random) -> +3 extra catches
  Top- 20: P@K=80% -> ~16 true decliners (vs ~11 random) -> +5 extra catches
  Top- 50: P@K=78% -> ~39 true decliners (vs ~27 random) -> +12 extra catches
  Top-100: P@K=75% -> ~75 true decliners (vs ~54 random) -> +21 extra catches
  Top-200: P@K=74% -> ~148 true decliners (vs ~108 random) -> +40 extra catches

  At 15 min/review, top-50 costs ~12.5 editor-hours.
  The queue is observed to surface more true decliners than random
  ordering at the same depth. Whether this justifies the review cost
  depends on the value of catching decline early -- a business decision,
  not a model decision.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### What gets exported

| File | Path | Git status |
|---|---|---|
| `playbook_action_queue.csv` | `work/outputs/` | **Not committed** (CI leak-guard blocks data; notebook regenerates it) |
| `playbook_metrics.json` | `work/outputs/` | Committed — the receipt your paper's numbers trace back to |
| `fig_precision_at_k.png` | `work/figures/` | Committed — reusable figure for the paper |
| `fig_reason_code_distribution.png` | `work/figures/` | Committed — reusable figure for the paper |

In [8]:
# ── Export 1: Ranked action queue CSV ──
queue_cols = [
    "model_rank", "content_id", "client_id", "model_prob", "baseline_score",
    "reason_code", "suggested_action",
    "impressions_90d", "avg_position", "days_since_last_update",
    "content_age_days", "ctr", "engagement_rate",
    "content_type", "main_intent", "is_declining",
]

# content_type and main_intent were one-hot encoded -- recover from original df
df_model["content_type"] = df["content_type"].values
df_model["main_intent"] = df["main_intent"].values

queue = df_model[queue_cols].sort_values("model_rank")

output_path = Path("../../work/outputs/playbook_action_queue.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)

print(f"Wrote action queue: {output_path}")
print(f"  Rows: {len(queue):,}")
print(f"  Columns: {list(queue_cols)}")
print()
print("  ! This CSV stays out of git by design (CI leak-guard).")
print("    The notebook regenerates it on each run.")

Wrote action queue: ..\..\work\outputs\playbook_action_queue.csv
  Rows: 30,000
  Columns: ['model_rank', 'content_id', 'client_id', 'model_prob', 'baseline_score', 'reason_code', 'suggested_action', 'impressions_90d', 'avg_position', 'days_since_last_update', 'content_age_days', 'ctr', 'engagement_rate', 'content_type', 'main_intent', 'is_declining']

  ! This CSV stays out of git by design (CI leak-guard).
    The notebook regenerates it on each run.


In [9]:
# ── Export 2: Playbook metrics JSON (committable receipt) ──
playbook_metrics = {
    "playbook_version": "w07_v1",
    "model": "XGBoost (n_estimators=200, max_depth=4)",
    "split": "GroupKFold by client_id, 5 folds",
    "seed": SEED,
    "n_rows": int(len(df)),
    "n_clients": int(df["client_id"].nunique()),
    "n_features": len(FEATURE_COLS),
    "base_rate": round(float(base_rate), 4),
    "oof_auc": round(float(oof_auc), 4),
    "fold_auc_range": [round(float(min(fold_aucs)), 3), round(float(max(fold_aucs)), 3)],
    "fold_auc_std": round(float(np.std(fold_aucs)), 4),
    "precision_at_k": {},
    "accuracy": round(float(acc), 4),
    "false_positives": int(fp),
    "false_negatives": int(fn),
    "reason_code_counts": df_model["reason_code"].value_counts().to_dict(),
    "action_counts": df_model["suggested_action"].value_counts().to_dict(),
}

for k in ks:
    playbook_metrics["precision_at_k"][f"P@{k}"] = round(
        float(precision_at_k(y, oof_xgb, k)), 4
    )

metrics_path = Path("../../work/outputs/playbook_metrics.json")
metrics_path.write_text(json.dumps(playbook_metrics, indent=2))
print(f"Wrote playbook metrics: {metrics_path}")
print(json.dumps(playbook_metrics, indent=2))

Wrote playbook metrics: ..\..\work\outputs\playbook_metrics.json
{
  "playbook_version": "w07_v1",
  "model": "XGBoost (n_estimators=200, max_depth=4)",
  "split": "GroupKFold by client_id, 5 folds",
  "seed": 42,
  "n_rows": 30000,
  "n_clients": 32,
  "n_features": 32,
  "base_rate": 0.5421,
  "oof_auc": 0.6939,
  "fold_auc_range": [
    0.605,
    0.706
  ],
  "fold_auc_std": 0.0387,
  "precision_at_k": {
    "P@10": 0.8,
    "P@20": 0.8,
    "P@50": 0.78,
    "P@100": 0.75,
    "P@200": 0.74
  },
  "accuracy": 0.6482,
  "false_positives": 5948,
  "false_negatives": 4606,
  "reason_code_counts": {
    "low_visibility": 13274,
    "position_slipping": 9148,
    "visible_only": 7561,
    "stale_and_slipping": 14,
    "stale_visible": 3
  },
  "action_counts": {
    "deprioritize": 13274,
    "optimize_position": 9148,
    "monitor": 7561,
    "refresh_and_reposition": 14,
    "refresh": 3
  }
}


In [10]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ── Export 3: Precision@K comparison figure ──
fig_dir = Path("../../work/figures")
fig_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))

ks_plot = [10, 20, 50, 100, 200]
p_baseline = [precision_at_k(y, baseline_scores, k) * 100 for k in ks_plot]
p_xgb = [precision_at_k(y, oof_xgb, k) * 100 for k in ks_plot]
p_random = [base_rate * 100] * len(ks_plot)

ax.plot(ks_plot, p_xgb, "o-", color="#2563eb", linewidth=2, markersize=8, label="XGBoost queue")
ax.plot(ks_plot, p_baseline, "s--", color="#f59e0b", linewidth=2, markersize=7, label="Rule baseline")
ax.plot(ks_plot, p_random, ":", color="#94a3b8", linewidth=1.5, label=f"Base rate ({base_rate*100:.1f}%)")

ax.set_xlabel("K (queue depth)", fontsize=12)
ax.set_ylabel("Precision@K (%)", fontsize=12)
ax.set_title("Ranked Queue Quality: Precision@K", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.set_ylim(40, 105)
ax.set_xticks(ks_plot)
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
fig_path = fig_dir / "fig_precision_at_k.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {fig_path}")

Saved: ..\..\work\figures\fig_precision_at_k.png


In [11]:
# ── Export 4: Reason code distribution figure ──
fig2, ax2 = plt.subplots(figsize=(8, 5))

reason_counts = df_model["reason_code"].value_counts()
colors = {
    "low_visibility": "#94a3b8",
    "position_slipping": "#f59e0b",
    "visible_only": "#22c55e",
    "stale_and_slipping": "#ef4444",
    "stale_visible": "#f97316",
}
bar_colors = [colors.get(code, "#64748b") for code in reason_counts.index]

ax2.barh(reason_counts.index, reason_counts.values, color=bar_colors, edgecolor="white")
ax2.set_xlabel("Number of pages", fontsize=12)
ax2.set_title("Reason Code Distribution (30k pages)", fontsize=14, fontweight="bold")

# Add count labels
for i, (code, count) in enumerate(reason_counts.items()):
    ax2.text(count + 200, i, f"{count:,}", va="center", fontsize=10)

ax2.invert_yaxis()
fig2.tight_layout()
fig_path2 = fig_dir / "fig_reason_code_distribution.png"
fig2.savefig(fig_path2, dpi=150, bbox_inches="tight")
plt.close(fig2)
print(f"Saved: {fig_path2}")

Saved: ..\..\work\figures\fig_reason_code_distribution.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Intended use + limits section explains who uses this and where it stops
- [x] Ranked actions have reason codes and archetype→action mapping
- [x] Human review rules state what a person must check before acting
- [x] No-go list states what should NEVER be automated (deletions, auto-refresh, causal claims)
- [x] Monitoring/retrain triggers are concrete and measurable
- [x] Cost/value analysis is honest and practical
- [x] Exports: queue CSV to `work/outputs/` (not committed), metrics JSON to `work/outputs/` (committed), figures to `work/figures/` (committed)
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.